# Week 63, Day 2: How properties work

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w063d2.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
class Temp:
    unit = "C"
    def __init__(self):
        self._c = 20
    @property
    def c(self):
        return self._c

t = Temp()
t.__dict__["c"] = 99
t.__dict__["unit"] = "F"
print(t.c, t.unit, Temp.unit)
try:
    t.c = 5
except AttributeError:
    print("can't set c")

In [ ]:
class Property:
    def __init__(self, fget=None, fset=None, fdel=None, doc=None):
        self.fget, self.fset, self.fdel = fget, fset, fdel
        self.__doc__ = doc if doc is not None else getattr(fget, "__doc__", None)

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        if self.fget is None:
            raise AttributeError("unreadable attribute")
        return self.fget(obj)

    def __set__(self, obj, value):
        if self.fset is None:
            raise AttributeError("can't set attribute")
        self.fset(obj, value)

    def __delete__(self, obj):
        if self.fdel is None:
            raise AttributeError("can't delete attribute")
        self.fdel(obj)

    def getter(self, fget):
        return type(self)(fget, self.fset, self.fdel, self.__doc__)

    def setter(self, fset):
        return type(self)(self.fget, fset, self.fdel, self.__doc__)

    def deleter(self, fdel):
        return type(self)(self.fget, self.fset, fdel, self.__doc__)

class Product:
    def __init__(self, price):
        self.price = price
    @Property
    def price(self):
        "The price in pence."
        return self._price
    @price.setter
    def price(self, value):
        if value <= 0:
            raise ValueError("price must be positive")
        self._price = value

p = Product(250)
p.price = 300
print(p.price, Product.price.__doc__)
try:
    p.price = -1
except ValueError as e:
    print(e)

In [ ]:
class DataDesc:
    def __get__(self, obj, objtype=None):
        return "data descriptor"
    def __set__(self, obj, value):
        raise AttributeError("read-only")

class NonData:
    def __get__(self, obj, objtype=None):
        return "non-data descriptor"

class Base:
    plain = "class attribute"
    a = DataDesc()

class Child(Base):
    b = NonData()

c = Child()
c.__dict__.update(a="instance", b="instance", plain="instance", own="instance")
print(c.a, "|", c.b, "|", c.plain, "|", c.own)
del c.__dict__["b"], c.__dict__["plain"]
print(c.b, "|", c.plain)

In [ ]:
class Counter:
    count = 0                         # a plain class attribute: a shared default

a, b = Counter(), Counter()
a.count += 1                          # reads Counter.count, then stores in a.__dict__
print(a.count, b.count, Counter.count, vars(a))
del a.count
print(a.count)
try:
    del a.count
except AttributeError:
    print("nothing left to delete on a")

class Shop:
    @property
    def open(self):
        return True
Shop.open = "replaced"                # setting on the class replaces the property
print(Shop().open)

In [ ]:
class StaticMethod:
    def __init__(self, func):
        self.func = func
    def __get__(self, obj, objtype=None):
        return self.func

class ClassMethod:
    def __init__(self, func):
        self.func = func
    def __get__(self, obj, objtype=None):
        cls = objtype if objtype is not None else type(obj)
        return self.func.__get__(cls, type(cls))

class Money:
    def __init__(self, pence):
        self.pence = pence
    @ClassMethod
    def from_pounds(cls, pounds):
        return cls(round(pounds * 100))
    @StaticMethod
    def valid(pence):
        return isinstance(pence, int) and pence >= 0

class Tip(Money):
    pass

t = Tip.from_pounds(1.5)
print(type(t).__name__, t.pence, Money.valid(5), t.valid(-1))

In [ ]:
class Account:
    def __init__(self, balance):
        self.balance = balance
    @property
    def balance(self):
        return self._balance
    @balance.setter
    def balance(self, value):
        if not isinstance(value, int):
            raise TypeError("pence must be an int")
        self._balance = value

class Savings(Account):
    MINIMUM = 100
    @Account.balance.setter
    def balance(self, value):
        if value < self.MINIMUM:
            raise ValueError("below the minimum balance")
        Account.balance.fset(self, value)          # the parent's checks too

s = Savings(500)
s.balance = 200
print(s.balance)
for bad in (50, "300"):
    try:
        s.balance = bad
    except (ValueError, TypeError) as e:
        print(type(e).__name__, e)

> **Key idea.** property is a data descriptor class holding fget, fset and fdel; getter, setter and deleter return new properties, so the setter must reuse the getter's name. Reading obj.name searches the class's MRO first: a data descriptor there wins; then the instance __dict__; then a non-data descriptor; then a plain class attribute; then __getattr__ or AttributeError. Setting uses a data descriptor's __set__ if there is one, and otherwise stores in the instance __dict__; setting on the class replaces the attribute. staticmethod's __get__ returns the plain function; classmethod's binds the class. In subclasses, extend properties with @Base.prop.setter or .getter and call Base.prop.fset; never let a getter read its own name.

### Exercise: Your own property

Write `Prop`, a property class: `Prop(fget=None, fset=None, fdel=None, doc=None)` with `__get__`, `__set__` and `__delete__` (raising `AttributeError` when the function is missing), the docstring taken from `fget` when `doc` isn't given, and `getter`, `setter` and `deleter` methods that return **new** `Prop` objects, leaving the original unchanged.

In [ ]:
class Prop:
    def __init__(self, fget=None, fset=None, fdel=None, doc=None):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_prop():\n    \"property from scratch\"\n    class Box:\n        def __init__(self):\n            self._w = 1\n        @Prop\n        def width(self):\n            \"The width.\"\n            return self._w\n        read_only = width\n        @width.setter\n        def width(self, v):\n            self._w = v\n        @width.deleter\n        def width(self):\n            self._w = 0\n    b = Box()\n    b.width = 5\n    assert b.width == 5 and Box.width.__doc__ == \"The width.\" and isinstance(Box.width, Prop)\n    del b.width\n    assert b.width == 0\n    assert Box.read_only is not Box.width and Box.read_only.fset is None\n    b.__dict__[\"width\"] = 99\n    assert b.width == 0\n    try:\n        Box.read_only.__set__(b, 3)\n        assert False\n    except AttributeError:\n        pass\n    try:\n        Prop().__get__(b, Box)\n        assert False\n    except AttributeError:\n        pass\n    assert Prop(doc=\"hi\").__doc__ == \"hi\"")

### Exercise: What kind of attribute?

Write `kind_of(cls, name)`: search `cls.__mro__` for the first class whose `__dict__` has `name` and return `"data descriptor"` (its type has `__set__` or `__delete__`), `"non-data descriptor"` (only `__get__`), or `"plain"`; return `"missing"` if no class has it. Check the attribute's **type**, not the attribute.

In [ ]:
def kind_of(cls, name):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_kinds():\n    \"classifying attributes\"\n    class A:\n        x = 1\n        @property\n        def p(self):\n            return 1\n        def m(self):\n            pass\n        s = staticmethod(len)\n        c = classmethod(lambda cls: cls)\n        __slots__ = (\"slot\",)\n    class B(A):\n        x = property(lambda self: 2)\n    assert kind_of(A, \"x\") == \"plain\" and kind_of(B, \"x\") == \"data descriptor\"\n    assert kind_of(B, \"p\") == \"data descriptor\" and kind_of(B, \"m\") == \"non-data descriptor\"\n    assert kind_of(A, \"s\") == \"non-data descriptor\" and kind_of(A, \"c\") == \"non-data descriptor\"\n    assert kind_of(A, \"slot\") == \"data descriptor\" and kind_of(A, \"nothing\") == \"missing\"\n    class Tricky:\n        thing = type(\"Obj\", (), {})()\n    Tricky.thing.__get__ = lambda *a: 1\n    assert kind_of(Tricky, \"thing\") == \"plain\"")

### Exercise: Static and class methods by hand

Write `static(func)` and `klass(func)` as descriptor classes behaving like `staticmethod` and `classmethod`: `static` returns the plain function whether read through the class or an instance; `klass` binds the class (the subclass, when read through a subclass or its instance). Use the function's own `__get__` to bind; don't use the built-ins or `types.MethodType`.

In [ ]:
class static:
    def __init__(self, func):
        ___

class klass:
    def __init__(self, func):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_binders():\n    \"staticmethod and classmethod\"\n    class Shape:\n        sides = 0\n        @klass\n        def describe(cls, extra=\"\"):\n            return f\"{cls.__name__} with {cls.sides} sides{extra}\"\n        @static\n        def area_units():\n            return \"cm2\"\n    class Square(Shape):\n        sides = 4\n    assert Shape.describe() == \"Shape with 0 sides\" and Square.describe(\"!\") == \"Square with 4 sides!\"\n    assert Square().describe() == \"Square with 4 sides\" and Shape.area_units() == \"cm2\" and Square().area_units() == \"cm2\"\n    assert Shape.describe.__self__ is Shape\n    assert not uses(source, \"staticmethod\") and not uses(source, \"classmethod\") and not uses(source, \"MethodType\")")

### Exercise: A property factory

Before writing descriptors, people often wrote **functions that return properties**. Write `checked(name, check, message)`: return a `property` that stores the value in `obj.__dict__["_" + name]`, and whose setter raises `ValueError(message)` when `check(value)` is false. Reading before setting raises `AttributeError`. Then use it to give a class `Person` a `name` (non-empty string) and an `age` (whole number from 0 to 150), set in `__init__(name, age)`.

In [ ]:
def checked(name, check, message):
    ___

class Person:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_factory():\n    \"properties from a function\"\n    p = Person(\"Ada\", 36)\n    assert p.name == \"Ada\" and p.age == 36 and vars(p) == {\"_name\": \"Ada\", \"_age\": 36}\n    for attr, bad in ((\"name\", \"\"), (\"age\", 151), (\"age\", -1), (\"age\", 2.5)):\n        try:\n            setattr(p, attr, bad)\n            assert False\n        except ValueError:\n            pass\n    assert isinstance(Person.__dict__[\"age\"], property)\n    class Thing:\n        size = checked(\"size\", lambda v: v > 0, \"size must be positive\")\n    t = Thing()\n    try:\n        t.size\n        assert False\n    except AttributeError:\n        pass\n    try:\n        t.size = 0\n    except ValueError as e:\n        assert str(e) == \"size must be positive\"")

### Exercise: Extending an inherited property

`Thermostat` has a `target` property that only accepts numbers. Write `SafeThermostat(Thermostat)` whose `target` setter **also** rejects values outside 5 to 30 (`ValueError`), reusing the parent's checks and storage through `Thermostat.target.fset`; and `LabelledThermostat(Thermostat)` whose `target` getter returns a string like `"21 C"`, keeping the parent's setter. Use `@Thermostat.target.setter` and `@Thermostat.target.getter`.

In [ ]:
class Thermostat:
    def __init__(self, target):
        self.target = target
    @property
    def target(self):
        return self._target
    @target.setter
    def target(self, value):
        if not isinstance(value, (int, float)):
            raise TypeError("target must be a number")
        self._target = value

class SafeThermostat(Thermostat):
    ___

class LabelledThermostat(Thermostat):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_inherit():\n    \"extending properties\"\n    s = SafeThermostat(20)\n    s.target = 25\n    assert s.target == 25\n    for bad, err in ((31, ValueError), (4, ValueError), (\"20\", TypeError)):\n        try:\n            s.target = bad\n            assert False\n        except err:\n            pass\n    lab = LabelledThermostat(21)\n    assert lab.target == \"21 C\"\n    lab.target = 18\n    assert lab.target == \"18 C\"\n    try:\n        lab.target = \"warm\"\n        assert False\n    except TypeError:\n        pass\n    assert Thermostat(40).target == 40")

### Bug hunt: A getter that reads itself

The getter returns `self.name`, which calls the getter again, forever, ending in `RecursionError`. Store and read the value under a different name so this prints `Ada`.

In [ ]:
class User:
    def __init__(self, name):
        self.name = name
    @property
    def name(self):
        return self.name
    @name.setter
    def name(self, value):
        self.__dict__["_name"] = value.strip()

print(User("  Ada ").name)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"no recursion\"\n    expect_output(\"Ada\")")

### Bug hunt: A setter with the wrong name

The setter is defined as `set_age`, so the complete property is stored under `set_age` while `age` is still the read-only one: `p.age = 30` raises `AttributeError`. Give the setter the same name as the getter so this prints `30`.

In [ ]:
class Person:
    @property
    def age(self):
        return self._age
    @age.setter
    def set_age(self, value):
        self._age = value

p = Person()
p.age = 30
print(p.age)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"same name\"\n    expect_output(\"30\")")

### Bug hunt: An override that lost the setter

The subclass redefines `level` with a plain `@property`, which replaces the parent's whole property, setter included, so `g.level = 3` raises `AttributeError`. Build the new getter from the parent's property so this prints `level 3`.

In [ ]:
class Game:
    @property
    def level(self):
        return self._level
    @level.setter
    def level(self, value):
        self._level = value

class LabelledGame(Game):
    @property
    def level(self):
        return f"level {self._level}"

g = LabelledGame()
g.level = 3
print(g.level)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"keep the setter\"\n    expect_output(\"level 3\")")

### Puzzle: A property with a checked setter

Arrange the lines of a class with a validated property. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # "data" or "non-data": a property with no setter
q = ___   # True or False: obj.x = 1 stores in obj.__dict__ when the class has a plain attribute x
r = ___   # what staticmethod's __get__ returns: "the function" or "a bound method"
s = ___   # True or False: the class is searched before the instance __dict__ when reading obj.name
t = ___   # c = classmethod(f); in Sub.method(), what's bound as the first argument: "Base" or "Sub"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == \"data\" and q is True and r == \"the function\", \"p, q and r\"\n    assert s is True and t == \"Sub\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: attribute lookup by hand

Write the attribute machinery yourself, using only `type`, `vars`, `__mro__`, `hasattr` on **types** and descriptor methods called on the descriptor's type (no `getattr`, `setattr` or `delattr` on the object itself):

- `get_attribute(obj, name)`: follow the six steps of the lookup order, including calling `type(obj).__getattr__(obj, name)` if the class defines `__getattr__` and the lookup fails; otherwise raise `AttributeError(name)`.
- `explain(obj, name)`: where the value would come from, as one of `"data descriptor in C"`, `"instance"`, `"non-data descriptor in C"`, `"class attribute in C"`, `"__getattr__"`, or `"missing"` (where `C` is the name of the class that defines it).
- `set_attribute(obj, name, value)`: a data descriptor's `__set__` if there is one, otherwise store in `vars(obj)`.
- `delete_attribute(obj, name)`: a data descriptor's `__delete__`, otherwise remove from `vars(obj)`, raising `AttributeError(name)` if it isn't there.

(Ignore `__slots__` and metaclasses; assume instances have a `__dict__`.)

In [ ]:
def get_attribute(obj, name):
    ___

def explain(obj, name):
    ___

def set_attribute(obj, name, value):
    ___

def delete_attribute(obj, name):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_lookup():\n    \"the lookup order\"\n    class NonData:\n        def __get__(self, obj, objtype=None):\n            return \"nd\"\n    class Base:\n        plain = \"class\"\n        nd = NonData()\n        @property\n        def p(self):\n            return \"prop\"\n        def method(self):\n            return \"called\"\n    class Child(Base):\n        pass\n    c = Child()\n    vars(c).update(p=\"inst\", nd=\"inst\", own=\"inst\")\n    assert get_attribute(c, \"p\") == \"prop\" and get_attribute(c, \"nd\") == \"inst\"\n    assert get_attribute(c, \"own\") == \"inst\" and get_attribute(c, \"plain\") == \"class\"\n    assert get_attribute(c, \"method\")() == \"called\"\n    assert explain(c, \"p\") == \"data descriptor in Base\" and explain(c, \"nd\") == \"instance\"\n    del vars(c)[\"nd\"]\n    assert get_attribute(c, \"nd\") == \"nd\" and explain(c, \"nd\") == \"non-data descriptor in Base\"\n    assert explain(c, \"plain\") == \"class attribute in Base\" and explain(c, \"zzz\") == \"missing\"\n    try:\n        get_attribute(c, \"zzz\")\n        assert False\n    except AttributeError:\n        pass\n    class Fallback:\n        def __getattr__(self, name):\n            return name.upper()\n    assert get_attribute(Fallback(), \"abc\") == \"ABC\" and explain(Fallback(), \"abc\") == \"__getattr__\"\n\ndef test_set_delete():\n    \"setting and deleting\"\n    class Guarded:\n        tag = \"default\"\n        @property\n        def v(self):\n            return self._v\n        @v.setter\n        def v(self, value):\n            self._v = value * 2\n        @v.deleter\n        def v(self):\n            self._v = None\n    g = Guarded()\n    set_attribute(g, \"v\", 5)\n    assert g.v == 10 and \"v\" not in vars(g)\n    set_attribute(g, \"tag\", \"mine\")\n    assert vars(g)[\"tag\"] == \"mine\" and Guarded.tag == \"default\"\n    delete_attribute(g, \"tag\")\n    assert g.tag == \"default\"\n    try:\n        delete_attribute(g, \"tag\")\n        assert False\n    except AttributeError:\n        pass\n    delete_attribute(g, \"v\")\n    assert g.v is None\n    assert not uses(source, \"getattr\") and not uses(source, \"setattr\") and not uses(source, \"delattr\")")

## Warm-up

The property beat the instance's own `c`, but the plain class attribute `unit` lost to the instance's. And a property without a setter refuses assignment rather than quietly storing a value on the instance.

## property in pure Python

`property` is a class, and `@property` just calls it with your getter. Here is an equivalent written in Python (the real one is in C but behaves the same way):

- It stores up to three functions: `fget`, `fset`, `fdel`, and a docstring.
- `__get__` calls `fget(obj)` (or returns itself through the class); `__set__` calls `fset(obj, value)`, raising `AttributeError` if there's none; `__delete__` does the same with `fdel`.
- `setter(func)` doesn't change the property: it returns a **new** property with the same getter and the new setter. That's why the setter method must have the **same name** as the getter: `@price.setter def price(...)` rebinds the name `price` to the new, complete property.

Because it defines `__set__` and `__delete__` even when they would only raise, a property is always a **data** descriptor, which is why it beat the instance `__dict__` in the warm-up.

## The lookup order

When you read `obj.name`, `object.__getattribute__` does this:

1. Look `name` up in the **class** and its bases, in MRO order (Week 30), stopping at the first class whose `__dict__` has it. Call what it finds `attr`.
2. If `attr` is a **data descriptor** (its type has `__set__` or `__delete__`), return `type(attr).__get__(attr, obj, type(obj))`.
3. Otherwise, if `name` is in **`obj.__dict__`**, return that value.
4. Otherwise, if `attr` is a **non-data descriptor** (its type has `__get__`), return `type(attr).__get__(attr, obj, type(obj))`.
5. Otherwise, if `attr` was found at all, return it as it is (a plain class attribute).
6. Otherwise raise `AttributeError`, and if the class defines `__getattr__`, Python calls `__getattr__(obj, name)` instead of raising (Day 3).

Two details: the descriptor methods are looked up on the descriptor's **type**, not on the descriptor object; and the class is searched **before** the instance, so that step 2 can win.

## Setting and deleting

Assignment `obj.name = value` (`object.__setattr__`) is simpler:

1. Look `name` up in the class and its bases. If it's a **data descriptor**, call its `__set__(obj, value)` (and that's where a read-only property raises).
2. Otherwise store the value in **`obj.__dict__`** (or in a slot, for `__slots__` classes), even if the class has a plain attribute or a non-data descriptor with that name.

`del obj.name` mirrors it: a data descriptor's `__delete__`, otherwise removal from `obj.__dict__` (and `AttributeError` if it isn't there, even if the class has the attribute).

Setting through the **class** (`Cls.name = value`) is different: it just replaces the class attribute, descriptor or not (the class's own metaclass decides, Thursday). That's how monkeypatching works.

## staticmethod and classmethod

Both are non-data descriptors whose `__get__` decides what gets bound:

- **`staticmethod(func)`**: `__get__` returns `func` itself, so nothing is bound and there's no `self`.
- **`classmethod(func)`**: `__get__` binds the **class** (`objtype`, or `type(obj)`) as the first argument, using the function's own descriptor: `func.__get__(cls, type(cls))`. Called on a subclass or its instance, it gets the subclass, which is what makes alternative constructors (Week 29) work with inheritance.

## Properties and inheritance

Properties are class attributes, so a subclass can override one. Two traps:

- **Defining `@property def price(self)` again in the subclass replaces the whole property**, setter included. If you only want a new getter, build from the parent's: `@Base.price.getter`. To extend only the setter, use `@Base.price.setter`, and call the parent's setter through the property: `Base.price.fset(self, value)` (`super().price = value` doesn't work, because `super()` objects don't support assignment through descriptors).
- **A property's getter can't call itself**: `return self.price` inside `price` recurses forever. Store under a different name (`self._price`).

Also remember that a property runs code on every access. Keep getters cheap and free of surprises; if a value is expensive and doesn't change, cache it (`functools.cached_property`, or yesterday's `Lazy`).

## Practice

## Quiz

## Daily challenge

## Before you go

Take one class you wrote in Phase 3 with properties and predict, for each attribute, which step of the lookup order supplies it. Tomorrow: `__getattr__`, `__getattribute__`, `__setattr__` and `__delattr__`, the hooks that let a class take over attribute access completely.